In [1]:
from pdfminer.utils import open_filename
from pydantic import BaseModel
from langdetect import detect
from langchain_core.prompts import ChatPromptTemplate
import uuid
from langchain.vectorstores import Chroma
from langchain.storage import InMemoryStore
from langchain.schema.document import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever
import tempfile
import pytesseract
import streamlit as st
import os
import base64
import chromadb
from langchain.docstore.in_memory import InMemoryDocstore
from unstructured.partition.pdf import partition_pdf
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama.llms import OllamaLLM
from base64 import b64decode
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

In [2]:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [ ]:
retrieved_tables = []
images = []
texts = ""
table_summaries = []
image_summaries = []
pdf_path = r"C:\Users\amans\Downloads\Structural Analysis by R. C. Hibbeler - 8th Edition-24-47.pdf"
chunks = partition_pdf(
    filename=pdf_path,
    extract_images_in_pdf=True,
    extract_images_block_types = ["Image"],# "Table"], # if want to extract tables as images, add "Table" aswell
    strategy = "hi_res",
    extract_image_block_to_payload = True, # if you want to have base64 as metadata which is required if you want to send the image to llm
    infer_table_structure=True, # to extract tables
    include_metadata = True,
)


for chunk in chunks:
    if "Table" in str(type(chunk)):
        retrieved_tables.append(chunk.metadata.text_as_html)
    elif "Image" in str(type(chunk)):
        images.append(str(chunk.metadata.image_base64))
    elif any(j in str(type(chunk)) for j in ["Text", "Title"]) and len(chunk.text) > 1:
         texts += str(chunk)


splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap = 120)
split_texts = splitter.split_text(texts)

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


In [4]:
llm = OllamaLLM(model = "llava:7b", base_url="http://127.0.0.1:11434",
            timeout=60,
            temperature=0.1,
            num_predict=512)
template = PromptTemplate(
    template=''' you will be given {input type}, you are required to summarise it and
    provide all the details in it along with the numerical values. no need to provide any additional comments or narration, just give the summary.
    Here is the input: {value}
    ''',
    input_variables=['input type', 'value']
)
for i in ['base64 string of images', 'html of tables']:
    if i == 'base64 string of images':
        for img in images:
            llm_with_image_context = llm.bind(images = [img]) 
            summary = llm_with_image_context.invoke('''summarise the image in detail and analyse all the graphs if present
          also include the numerical values that can be inferred.
          Do not give any additional comments or ask for any other data''')
            image_summaries.append(summary)
    if i == 'html of tables':
        for tbl in retrieved_tables:
            prompt = template.invoke({'input type':i, 'value':tbl})
            summary = llm.invoke(prompt)
            table_summaries.append(summary)

In [5]:
model_name = "BAAI/bge-base-en-v1.5"
docstore = InMemoryStore()
vectorstore = Chroma(collection_name="RAG_chatbot",
                     persist_directory=r"C:\Users\amans\Desktop\Chatbot_project\chromadb",
                embedding_function=HuggingFaceEmbeddings(model_name=model_name))
id_key = "doc_id"
retriever = MultiVectorRetriever(
                vectorstore=vectorstore,
                docstore=docstore,
                id_key=id_key,
        )


doc_ids = [str(uuid.uuid4()) for _ in split_texts]
# Store original text documents in docstore
text_docs = [
    Document(page_content=content, metadata={"data_type": "text"})
    for content in split_texts
]
retriever.docstore.mset(list(zip(doc_ids, text_docs)))
# Add text documents to vectorstore with doc_id metadata
final_texts = [
    Document(page_content=content, metadata={id_key: doc_ids[i], "data_type": "text"})
    for i, content in enumerate(split_texts)
]
retriever.vectorstore.add_documents(final_texts)

table_ids = [str(uuid.uuid4()) for _ in table_summaries]
table_docs = [
        Document(page_content=t, metadata = {"data_type": "table"})
        for t in retrieved_tables
        ]
retriever.docstore.mset(list(zip(table_ids, table_docs)))

table_summaries_doc = [
         Document(page_content=summary, metadata={id_key: table_ids[i], "data_type": "table_sum"})
         for i, summary in enumerate(table_summaries)
     ]
retriever.vectorstore.add_documents(table_summaries_doc)

img_ids = [str(uuid.uuid4()) for _ in images]
img_docs = [
     Document(page_content = img, metadata = {"data_type":"image"})
     for img in images
]
retriever.docstore.mset(list(zip(img_ids,img_docs)))

image_summaries_doc = [
                 Document(page_content=summary, metadata={id_key: img_ids[i], "data_type":"image_sum"}) 
                 for i, summary in enumerate(image_summaries)
]
retriever.vectorstore.add_documents(image_summaries_doc)

C:\Users\amans\AppData\Local\Temp\ipykernel_21728\3081297261.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(collection_name="RAG_chatbot",


['f73f4169-29ba-4698-b5a5-10cd54c9dcbe',
 '6cfd7d56-1801-4e33-a7cd-0964c4c9c698',
 'e8cd5240-aae0-459c-b468-1df2c4f15b55',
 '2958b48f-ba25-4a4c-8165-ce11c755359c',
 'cd48d871-fee7-4a35-816e-d04877984529',
 '4e5210d3-7991-4940-8d9e-79b2c3e7381b',
 '377721ba-737a-4012-b0b2-3d1c49d0be21',
 '8726dcdf-05cc-430f-8e1b-b43487ddbddf',
 'a0158a67-e565-4693-bcef-805fec5f0cd0',
 '892f154b-70ad-40b7-b0d3-b5a8c3976d8c',
 '3c1a1452-c2f1-4074-a2b3-d1bd9b43ce56',
 'bbaf5dcb-c056-4a68-8120-959851722d7f',
 '5e43f3d7-a568-4982-a9c3-2c7fd550c5bf',
 '81ab50de-5609-4179-9e7b-f2b624536c83',
 'bd323349-d548-4132-b083-783c33244892',
 '47e24688-58b7-4a91-9ea6-85289637eea9',
 'c2ccd2de-7943-4f01-a853-a6d640db3364',
 '5a2a221f-6b50-4dda-9448-795739ff4efd',
 '82ca80da-7b2f-411d-8973-66af56dce039',
 'ca465d06-a62e-45b0-bee5-0e4b620723d4',
 'aaabebdb-449e-4ace-947e-1280142554d1',
 'f0c3fb80-bdb7-422a-87a7-d6afc088b6e4',
 '3df74ec4-8e84-48bf-9ad3-a3e94af47efb',
 '3cb0cac3-86a9-41ec-8dfa-ea3eaab1c405',
 '4df302ad-624b-

In [1]:
from bs4 import BeautifulSoup
from tabulate import tabulate

def print_html_table(html):
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")

    if not tables:
        print("No tables found in the HTML.")
        return

    for index, table in enumerate(tables, start=1):
        headers = []
        rows = []

        # Get headers if available
        header_row = table.find("tr")
        if header_row:
            headers = [th.get_text(strip=True) for th in header_row.find_all("th")]

        # Get all rows (skip header if already taken)
        for tr in table.find_all("tr")[1 if headers else 0:]:
            row = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
            rows.append(row)

        print(f"\nTable {index}:")
        print(tabulate(rows, headers=headers if headers else "firstrow", tablefmt="grid"))

import base64
from io import BytesIO
from PIL import Image

def show_image_from_base64(base64_str):
    try:
        # Decode base64 string
        image_data = base64.b64decode(base64_str)
        image = Image.open(BytesIO(image_data))

        # Show the image
        image.show()

    except Exception as e:
        print("Error displaying image:", e)

In [ ]:
from langchain.schema.runnable import Runnable
from typing import Dict, Any, List

class MultimodalQAChain(Runnable):
    def __init__(self, retriever, llm, qa_prompt):
        self.retriever = retriever
        self.llm = llm
        self.qa_prompt = qa_prompt
    
    def invoke(self, input_data: Dict[str, Any]) -> str:
        question = input_data["question"]
        chat_history = input_data.get("chat_history", [])
        
        # Retrieve documents
        docs = self.retriever.get_relevant_documents(question)
        # docs_by_type = self.parse_docs(docs)
        
        # Build text context (WITHOUT images)
        text_context = self.context(docs)
        
        # Format the prompt
        messages = self.qa_prompt.format_messages(
            question=question,
            context=text_context,
            chat_history=chat_history
        )
        response = self.llm.invoke(messages)
        
        # Handle images separately
        # images = docs_by_type.get("images", [])
        # if images:
        #     # Bind images to LLM - THIS IS THE KEY FIX
        #     llm_with_images = self.llm.bind(images=images)
        #     response = llm_with_images.invoke(messages)
        # else:
        #     response = self.llm.invoke(messages)
        
        return response
    
    def context(docs):
        context = []
        for doc in docs:
            content = str(doc.page_content) if hasattr(doc, "page_content") else str(doc)
            if doc.metadata["data_type"] == "table":
                context.append("\n\n[Table Content]:\n" + content)
            elif doc.metadata["data_type"] == "image":
                context.append({"type": "image_url", "image_url": {"url": "data:image/png;base64," + content}})
            else:
                context.append("\n\n[Text Content]:\n" + content)
        return context
    
    def parse_docs(self, docs):
        retrieved_images = []
        retrieved_table = []
        retrieved_texts = []
        
        for doc in docs:
            content = str(doc.page_content) if hasattr(doc, "page_content") else str(doc)
            if doc.metadata["data_type"] == "table":
                retrieved_table.append(content)
            elif doc.metadata["data_type"] == "text":
                retrieved_texts.append(content)
            elif doc.metadata["data_type"] == "image_sum":
                retrieved_texts.append(content)
            elif doc.metadata["data_type"] == "table_sum":
                retrieved_texts.append(content)
            elif doc.metadata["data_type"] == "image":
                retrieved_images.append(content)
        for txt in retrieved_texts:
            print(f"retrieved text:{txt}\n")
        for tab in retrieved_table:
            print_html_table(tab)
            print('\n')
        for im in retrieved_images:
            show_image_from_base64(im)
            print('\n')
        return {"texts": retrieved_texts, "images": retrieved_images, "tables": retrieved_table}
    
    def build_text_context(self, docs_by_type):
        context_text = ""
        if docs_by_type.get("texts"):
            context_text += "\n\n[Text Content]:\n" + "\n".join(docs_by_type["texts"])
        if docs_by_type.get("tables"):
            context_text += "\n\n[Table Content]:\n" + "\n".join(docs_by_type["tables"])
        return context_text

NameError: name 'ChatPromptTemplate' is not defined

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Strictly only use the following context (text, table, or image summary) to answer the user's question. "
               "If the answer is not in the context, say 'I don't know based on the provided document. "
               "Convert all HTMLs to tables to infer from it also.'"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
    ("system", "Context:\n{context}")
])

# Create the custom QA chain
qa_chain = MultimodalQAChain(retriever, llm, qa_prompt)

chat_history = []
user_question = input("enter query:")
response = qa_chain.invoke({
            "question": user_question,
            "chat_history": chat_history
        })
chat_history.append(HumanMessage(content=user_question))
chat_history.append(AIMessage(content=response))